In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [3]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [4]:
dataset_path = r"C:\Users\LENOVO\Downloads\archive\plantvillage dataset\color"

full_dataset = datasets.ImageFolder(dataset_path, transform=train_transform)

print("Total images:", len(full_dataset))
print("Classes:", full_dataset.classes)

Total images: 54305
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spid

In [5]:
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_data, valid_data = random_split(full_dataset, [train_size, valid_size])
valid_data.dataset.transform = valid_transform

In [6]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)

num_classes = len(full_dataset.classes)
print("Number of classes:", num_classes)

Number of classes: 38


In [9]:
model = models.resnet50(pretrained=True)

In [10]:

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

In [11]:
num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes)
)
model = model.to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [13]:
epochs = 25

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        best_acc = 0


    train_acc = 100 * correct / total

   
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step()

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss {train_loss/len(train_loader):.4f} | "
          f"Train Acc {train_acc:.2f}% | "
          f"Val Acc {val_acc:.2f}%")

Epoch 1/25 | Train Loss 0.3226 | Train Acc 91.44% | Val Acc 98.63%
Epoch 2/25 | Train Loss 0.0541 | Train Acc 98.38% | Val Acc 98.62%
Epoch 3/25 | Train Loss 0.0380 | Train Acc 98.92% | Val Acc 98.98%
Epoch 4/25 | Train Loss 0.0329 | Train Acc 98.99% | Val Acc 98.91%
Epoch 5/25 | Train Loss 0.0260 | Train Acc 99.23% | Val Acc 98.99%
Epoch 6/25 | Train Loss 0.0069 | Train Acc 99.81% | Val Acc 99.61%
Epoch 7/25 | Train Loss 0.0041 | Train Acc 99.89% | Val Acc 99.67%
Epoch 8/25 | Train Loss 0.0033 | Train Acc 99.91% | Val Acc 99.57%
Epoch 9/25 | Train Loss 0.0024 | Train Acc 99.94% | Val Acc 99.70%


KeyboardInterrupt: 

In [14]:
torch.save(model.state_dict(), "model_epoch9.pth")